In [3]:
from pathlib import Path
import pandas as pd

pasta_atual = Path.cwd().resolve()
print(f"O Jupyter está rodando na pasta: {pasta_atual}")

BASE_DIR = pasta_atual.parent.parent

input_file = BASE_DIR / "data" / "raw" / "conectatel-dados" / "log_chamados" / "log_chamados_sintetico.csv"
output_file = BASE_DIR / "data" / "processed" / "log_chamados" / "chamados_clean.csv"

print(f"\nTentando LER de: {input_file}")
print(f"Tentando SALVAR em: {output_file}")


O Jupyter está rodando na pasta: C:\Users\USUARIO\Downloads\Rag-ConectaTel\src\01_pipeline_tratamento

Tentando LER de: C:\Users\USUARIO\Downloads\Rag-ConectaTel\data\raw\conectatel-dados\log_chamados\log_chamados_sintetico.csv
Tentando SALVAR em: C:\Users\USUARIO\Downloads\Rag-ConectaTel\data\raw\conectatel-dados\log_chamados\chamados_clean.csv
O arquivo ORIGINAL existe? True


In [4]:
MAPA_BOOLEANO = {
    "sim": True,
    "s": True,
    "1": True,
    "true": True,
    "não": False,
    "nao": False,
    "n": False,
    "0": False,
    "false": False,
}

MAPA_ESTADOS = {
    "são paulo": "SP",
    "sao paulo": "SP",
    "sp": "SP",
    "rio de janeiro": "RJ",
    "rj": "RJ",
    "minas gerais": "MG",
    "mg": "MG",
    "bahia": "BA",
    "ba": "BA",
    "paraná": "PR",
    "parana": "PR",
    "pr": "PR",
    "rio grande do sul": "RS",
    "rs": "RS",
    "CEARÁ": "CE",
    "ceará": "CE",
    "ce": "CE",
    "PERNAMBUCO": "PE",
    "pernambuco": "PE",
    "pe": "PE",
}


In [5]:
def tratar_booleano(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    return MAPA_BOOLEANO.get(val_str, None)

In [6]:
def tratar_estado(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    return MAPA_ESTADOS.get(val_str, str(val).strip().upper())

In [7]:
def processar_log_chamados(caminho_entrada: Path, caminho_saida: Path):
    df = pd.read_csv(caminho_entrada, encoding="utf-8")
    df = df.drop_duplicates()

    if "data_abertura" in df.columns:
        df["data_abertura"] = pd.to_datetime(
            df["data_abertura"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    cols_texto = [
        "canal",
        "categoria",
        "subcategoria",
        "cidade",
        "plano_atual",
        "resumo_atendimento",
    ]
    for col in cols_texto:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            if col in ["canal", "categoria", "cidade", "plano_atual"]:
                df[col] = df[col].str.title()
            df[col] = df[col].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

    if "estado" in df.columns:
        df["estado"] = df["estado"].apply(tratar_estado)

    cols_bool = ["resolvido_primeiro_contato", "encaminhado_humano"]
    for col in cols_bool:
        if col in df.columns:
            df[col] = df[col].apply(tratar_booleano)

    if "duracao_minutos" in df.columns:
        df["duracao_minutos"] = pd.to_numeric(
            df["duracao_minutos"], errors="coerce"
        )
        df.loc[
            (df["duracao_minutos"] <= 0) | (df["duracao_minutos"] > 180),
            "duracao_minutos",
        ] = None

    if "satisfacao_1_a_5" in df.columns:
        df["satisfacao_1_a_5"] = pd.to_numeric(
            df["satisfacao_1_a_5"], errors="coerce"
        )
        df.loc[~df["satisfacao_1_a_5"].between(1, 5), "satisfacao_1_a_5"] = None

    caminho_saida.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(caminho_saida, index=False, encoding="utf-8")

    return df

In [8]:
if input_file.exists():
    df_tratado = processar_log_chamados(input_file, output_file)
    print("Tratamento concluído!")
    display(df_tratado.head())
else:
    print("ERRO: O arquivo ainda não foi encontrado no caminho acima.")

Tratamento concluído!


,chamado_id,data_abertura,canal,categoria,subcategoria,estado,cidade,duracao_minutos,resolvido_primeiro_contato,encaminhado_humano,satisfacao_1_a_5,plano_atual,resumo_atendimento
0,CHM-10004,2026-08-21,Telefone,Cobertura,Sinal instavel,RJ,Duque De Caxias,12.0,True,False,4.0,Conecta Basico,Cliente relata quedas frequentes de sinal dura...
1,CHM-10259,2026-08-10,App,Portabilidade,Duvida sobre prazo de portabilidade,PR,Curitiba,9.0,True,False,4.0,Conecta Basico,Cliente pergunta quanto tempo leva o processo ...
2,CHM-10252,NaN,App,Outros,Solicitacao de fatura detalhada,BA,Feira De Santana,10.0,True,False,3.0,Conecta Plus,Cliente pede detalhamento de consumo por linha...
3,CHM-10146,2026-08-12,App,Cancelamento,Cancelamento de linha,BA,Feira De Santana,34.0,True,False,3.0,Conecta Basico,Cliente solicita encerramento de uma linha do ...
4,CHM-10020,2026-07-13,Loja Fisica,Outros,Elogio,MG,Belo Horizonte,26.0,True,False,4.0,Conecta Basico,Cliente elogia atendimento recebido em contato...
